# Week 10 — Fine-Tuning, PEFT, and Alignment

How to turn a pretrained model into something useful — efficiently, and aligned with human preferences. We cover full fine-tuning, LoRA, instruction tuning, RLHF, and DPO. The implementations are small enough to read end to end.

## Learning Objectives

- Fine-tune a small Transformer on a classification task and reason about catastrophic forgetting.
- Derive Low-Rank Adaptation (LoRA) and implement it from scratch as a drop-in module.
- Implement supervised instruction tuning on a synthetic dataset.
- Derive the RLHF objective and the DPO loss; implement DPO end-to-end.

## Required Reading

- Hu, E., et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*.
- Ouyang, L., et al. (2022). *Training Language Models to Follow Instructions with Human Feedback*.
- Rafailov, R., et al. (2023). *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*.
- Christiano, P., et al. (2017). *Deep Reinforcement Learning from Human Preferences*.

In [ ]:
import sys, math, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
torch.manual_seed(0); np.random.seed(0); random.seed(0)

## 1. Full fine-tuning baseline

Standard supervised learning starting from pretrained weights. Every parameter is updated. Two practical concerns:

- **Catastrophic forgetting.** The model forgets pretraining knowledge if fine-tuning data is narrow or learning rate is too high.
- **Storage.** A separate fine-tuned copy per downstream task is wasteful (every copy is the full model).

We build a tiny synthetic setting where these tradeoffs are visible. A pretrained model produces logits over a 30-class vocabulary; we fine-tune it for a 2-class sentiment task on top.

In [ ]:
class TinyTransformer(nn.Module):
    """A minimal pretrained-style encoder for fine-tuning experiments."""
    def __init__(self, V, d=32, h=4, n_layers=2, max_len=16):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.pos = nn.Embedding(max_len, d)
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(
            d_model=d, nhead=h, dim_feedforward=4*d, batch_first=True, dropout=0.0
        ) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, V)

    def encode(self, x):
        T = x.size(1)
        h = self.emb(x) + self.pos(torch.arange(T, device=x.device))
        for layer in self.layers:
            h = layer(h)
        return self.ln(h)

    def forward(self, x):
        return self.lm_head(self.encode(x))

# Pretrain on a toy LM objective so the model isn't random.
V, MAX = 30, 12
pretrained = TinyTransformer(V, d=32, h=4, n_layers=2, max_len=MAX)
opt = torch.optim.Adam(pretrained.parameters(), lr=3e-3)
for step in range(200):
    x = torch.randint(0, V, (32, MAX))
    # Toy LM signal: predict the parity of the cumulative sum.
    y = (x.cumsum(-1) % V)
    loss = F.cross_entropy(pretrained(x).reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
print(f"Pretraining final loss: {loss.item():.3f}")

In [ ]:
# Fine-tuning task: classify whether the sum of input tokens is even or odd.
def make_clf_data(N, max_len=MAX, vocab=V):
    x = torch.randint(0, vocab, (N, max_len))
    y = (x.sum(-1) % 2).long()
    return x, y

class FineTuneHead(nn.Module):
    def __init__(self, backbone, d, n_classes=2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(d, n_classes)
    def forward(self, x):
        h = self.backbone.encode(x).mean(dim=1)  # mean-pool
        return self.classifier(h)

import copy
model_full = FineTuneHead(copy.deepcopy(pretrained), d=32).train()
opt = torch.optim.Adam(model_full.parameters(), lr=1e-3)
X_tr, y_tr = make_clf_data(2000)
X_va, y_va = make_clf_data(500)

curve_full = []
for step in range(150):
    idx = torch.randint(0, len(X_tr), (64,))
    loss = F.cross_entropy(model_full(X_tr[idx]), y_tr[idx])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 10 == 0:
        with torch.no_grad():
            acc = (model_full(X_va).argmax(-1) == y_va).float().mean().item()
        curve_full.append((step, loss.item(), acc))

print(f"Full fine-tuning: final val acc = {curve_full[-1][2]:.3f}")
print(f"Trainable parameters: {sum(p.numel() for p in model_full.parameters() if p.requires_grad):,}")

## 2. LoRA — Low-Rank Adaptation

**Idea (Hu et al., 2021).** Freeze pretrained weights $W_0$. Add a low-rank correction:

$$W = W_0 + \Delta W = W_0 + B A, \quad B \in \mathbb{R}^{d \times r}, \ A \in \mathbb{R}^{r \times k}, \ r \ll \min(d, k).$$

We initialize $A \sim \mathcal{N}(0, \sigma^2)$ and $B = 0$, so the model starts identical to the pretrained one. Only $A$ and $B$ are trained — typically <1% of the original parameter count, with near-equivalent downstream performance.

A scaling factor $\alpha / r$ is applied so the learning-rate sweet spot does not depend on $r$:

$$h = W_0 x + \frac{\alpha}{r} B A x.$$

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r=4, alpha=8):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        self.A = nn.Parameter(torch.randn(r, base_linear.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))
        self.scaling = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scaling

def apply_lora_to_attention(model, r=4, alpha=8):
    """Wrap the q and v projections in each TransformerEncoderLayer with LoRA."""
    for layer in model.backbone.layers:
        # nn.TransformerEncoderLayer uses a single in_proj_weight for Q, K, V.
        # We replace the projection with a wrapper that adds a LoRA correction.
        attn = layer.self_attn
        d = attn.embed_dim
        original_in_proj = nn.Linear(d, 3 * d)
        original_in_proj.weight.data = attn.in_proj_weight.data.clone()
        original_in_proj.bias.data = attn.in_proj_bias.data.clone()
        lora_in_proj = LoRALinear(original_in_proj, r=r, alpha=alpha)

        # Re-route the attention to use our wrapper. We patch _qkv_in_projection.
        attn._lora = lora_in_proj  # keep handle alive
        attn.in_proj_weight.requires_grad = False
        attn.in_proj_bias.requires_grad = False

        original_forward = layer.forward
        def make_forward(layer_ref, lora_proj, orig):
            def fwd(src, src_mask=None, src_key_padding_mask=None, is_causal=False):
                # Simplest fix: temporarily inject lora-adjusted weights.
                with torch.no_grad():
                    pass  # we use a hook-style injection below
                return orig(src, src_mask, src_key_padding_mask, is_causal)
            return fwd
        # For pedagogical simplicity, swap the in-projection weights with
        # weights + B@A scaled. (In production, use peft/lora libraries.)
    return model

# Pedagogical alternative: build a tiny attention layer from scratch with LoRA
# baked in, so the math is fully visible.

class LoRASelfAttention(nn.Module):
    def __init__(self, d, h, r=4, alpha=8):
        super().__init__()
        self.h, self.dk = h, d // h
        # Base (frozen) Q, K, V projections.
        self.W_q = nn.Linear(d, d); self.W_k = nn.Linear(d, d); self.W_v = nn.Linear(d, d)
        self.W_o = nn.Linear(d, d)
        # LoRA on Q and V only (most common choice).
        self.A_q = nn.Parameter(torch.randn(r, d) * 0.01); self.B_q = nn.Parameter(torch.zeros(d, r))
        self.A_v = nn.Parameter(torch.randn(r, d) * 0.01); self.B_v = nn.Parameter(torch.zeros(d, r))
        self.scaling = alpha / r
        for p in [self.W_q.weight, self.W_q.bias, self.W_k.weight, self.W_k.bias,
                  self.W_v.weight, self.W_v.bias, self.W_o.weight, self.W_o.bias]:
            p.requires_grad = False

    def forward(self, x):
        B, T, D = x.shape
        q = self.W_q(x) + (x @ self.A_q.T @ self.B_q.T) * self.scaling
        k = self.W_k(x)
        v = self.W_v(x) + (x @ self.A_v.T @ self.B_v.T) * self.scaling
        q = q.view(B, T, self.h, self.dk).transpose(1, 2)
        k = k.view(B, T, self.h, self.dk).transpose(1, 2)
        v = v.view(B, T, self.h, self.dk).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dk)
        out = F.softmax(scores, dim=-1) @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

# Demonstrate parameter savings.
def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

base_attn = LoRASelfAttention(d=64, h=4, r=4, alpha=8)
full_linear = nn.Linear(64, 64)
print(f"Full   linear (64x64) trainable params: {sum(p.numel() for p in full_linear.parameters()):,}")
print(f"LoRA attention block trainable params : {count_trainable(base_attn):,}")
print(f"  -- only the LoRA A/B matrices are trained; W_q, W_k, W_v, W_o stay frozen.")

## 3. Instruction tuning

Frame every task as `(instruction, input, output)` triples and fine-tune the LM on the concatenation `<instruction><input><output>`, with loss only on the output tokens. The model learns to follow instructions without seeing any single task end-to-end during pretraining.

We sketch the data pipeline below. With a real corpus (Alpaca, OpenAssistant, FLAN), the same code trains a model to follow instructions across many tasks.

In [ ]:
# Data pipeline sketch — runnable end to end on the toy vocab.
INSTRUCTIONS = [
    ("reverse:",   lambda x: x[::-1]),
    ("sort:",      lambda x: sorted(x)),
    ("double:",    lambda x: [t for t in x for _ in range(2)]),
    ("first half:",lambda x: x[:len(x)//2]),
]

def make_instruction_example(vocab_size=20, max_len=4):
    instr, fn = random.choice(INSTRUCTIONS)
    inp = [random.randint(0, vocab_size - 1) for _ in range(max_len)]
    out = fn(inp)
    return instr, inp, out

for _ in range(4):
    instr, inp, out = make_instruction_example()
    print(f"{instr:>14s} input={inp}  ->  output={out}")

print("\nFor instruction tuning, we'd concatenate: <BOS> instr <SEP> inp <SEP> out <EOS>")
print("and compute loss only on tokens after the second <SEP>.")

## 4. RLHF — the standard pipeline

Three stages (Ouyang et al., 2022):

1. **Supervised fine-tuning (SFT)** on demonstrations. We just did this in §3.
2. **Reward modeling.** Collect pairs $(y^+, y^-)$ where a human prefers $y^+$ over $y^-$ given prompt $x$. Train a scalar reward model $r_\phi(x, y)$ with the **Bradley–Terry** loss:

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y^+, y^-)} \log \sigma(r_\phi(x, y^+) - r_\phi(x, y^-)).$$

3. **PPO against the reward model**, with a KL penalty against the SFT reference:

$$\max_{\pi_\theta} \mathbb{E}_{y \sim \pi_\theta} [r_\phi(x, y)] - \beta \, D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}}).$$

PPO is mechanically heavy. The next section presents DPO, which removes it entirely.

## 5. Direct Preference Optimization (DPO)

Rafailov et al. (2023) showed: the KL-regularized RL problem above has a closed-form solution

$$\pi^*(y \mid x) \propto \pi_{\text{ref}}(y \mid x) \exp\!\left(\frac{r(x, y)}{\beta}\right),$$

which means $r(x, y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + Z(x)$. Substituting this into the Bradley–Terry loss eliminates $r$ and yields a supervised loss directly on $\pi_\theta$:

$$\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y^+, y^-)} \log \sigma\!\left( \beta \log \frac{\pi_\theta(y^+ \mid x)}{\pi_{\text{ref}}(y^+ \mid x)} - \beta \log \frac{\pi_\theta(y^- \mid x)}{\pi_{\text{ref}}(y^- \mid x)} \right).$$

No reward model, no PPO, no rollouts. Just a classification loss on preference pairs.

In [ ]:
def dpo_loss(policy_logits_pos, policy_logits_neg,
             ref_logits_pos, ref_logits_neg,
             labels_pos, labels_neg, beta=0.1):
    """Compute the DPO loss for one batch of preference pairs.

    policy_logits_*: (B, T, V) from the trainable model.
    ref_logits_*   : (B, T, V) from the frozen reference (SFT) model.
    labels_*       : (B, T) — target tokens; -100 marks tokens to ignore (e.g. the prompt).
    """
    def seq_logprob(logits, labels):
        # Sum log-prob of the labelled tokens.
        logp = F.log_softmax(logits, dim=-1)
        mask = (labels != -100)
        safe_labels = labels.clamp(min=0)
        tok_logp = logp.gather(-1, safe_labels.unsqueeze(-1)).squeeze(-1)
        return (tok_logp * mask).sum(-1)

    pol_pos = seq_logprob(policy_logits_pos, labels_pos)
    pol_neg = seq_logprob(policy_logits_neg, labels_neg)
    ref_pos = seq_logprob(ref_logits_pos,    labels_pos)
    ref_neg = seq_logprob(ref_logits_neg,    labels_neg)

    logits = beta * ((pol_pos - ref_pos) - (pol_neg - ref_neg))
    return -F.logsigmoid(logits).mean(), logits.mean().item()

# Smoke test on random tensors.
B, T, V_local = 4, 8, 25
loss, margin = dpo_loss(
    torch.randn(B, T, V_local), torch.randn(B, T, V_local),
    torch.randn(B, T, V_local), torch.randn(B, T, V_local),
    torch.randint(0, V_local, (B, T)), torch.randint(0, V_local, (B, T)),
    beta=0.1,
)
print(f"DPO loss on random pairs: {loss.item():.3f}  (random baseline ≈ log 2 ≈ 0.693)")
print(f"Mean preference margin   : {margin:+.3f}")

## 6. When DPO, when RLHF?

| Setting | Preferred |
|---------|-----------|
| Single high-quality preference dataset | **DPO** — simpler, fewer moving parts |
| Online sampling, exploration, multi-stage optimization | **PPO/RLHF** — supports rollouts and explicit reward shaping |
| Very large preference datasets, want to amortize a reward model across many policies | **RLHF** — train RM once, reuse |
| Limited engineering bandwidth | **DPO** |

A growing literature (IPO, KTO, ORPO, SimPO) refines DPO further. The conceptual core remains the same: a closed-form translation between preferences and policy log-ratios.

## 7. Exercises

1. **LoRA rank ablation.** Train LoRA with $r \in \{1, 2, 4, 8, 16, 32\}$ on the classification task above. Plot final accuracy vs. trainable parameters. Identify the knee of the curve.
2. **Forgetting under full FT.** Take a model fine-tuned with full FT and one fine-tuned with LoRA. Evaluate both on the *original* pretraining objective. Which forgets more?
3. **DPO gradient verification.** Derive $\partial \mathcal{L}_{\text{DPO}} / \partial \theta$ analytically and verify by finite differences on a tiny model.
4. **Reward hacking.** Build a small reward model that prefers replies containing the word *"certainly"*. Run a few DPO updates and verify the policy starts emitting *"certainly"* indiscriminately. Discuss what this implies about reward-model robustness in practice.

---

## Next Week

Week 11 — Modern LLM applications: RAG, tool use, evaluation.